# Build the Text Training Table (`text_master.pkl`)

Joins the stock-day sentence embeddings from
`01 - feature extraction/features_08_text_embeddings.ipynb` with the keys, targets and
existing features of `merged_master.pkl`, and writes **one compact table with one row per
training stock-day that has message text** (about 3.5M rows). Nothing is fitted, compressed or
imputed here.

**Why a separate table, not extra columns on `merged_master`.** `merged_master` is the full
CRSP panel (15.5M stock-days), of which only ~23% have any message; attaching 384 float32
columns to all of it costs ~24 GB before any model notebook copies it. The text-only models can
only be estimated on stock-days that have text anyway, so they read this table instead.

**Contents.** Every column of `merged_master` (keys, `f_cumret1`, the `ar_*` abnormal returns,
the 53 StockTwits features), plus `mm_index` (the row label of the same stock-day in
`merged_master`, which the prediction files carry as `index`), `embed_n` (message-symbol pairs
behind the text) and `embed_000..embed_383` (mean-pooled, L2-normalised message vectors).

**Sample.** Rows are `merged_master` rows, so a ticker that maps to two PERMNOs on a date
appears twice with identical features and text, exactly as in the baseline models. Stock-days
with messages but no text (2 rows) are dropped; embedding stock-days outside the CRSP panel
(2024, and tickers absent from `merged_master`) are dropped.

Consumers: `03a - linear regression/prediction_linear_regression_text_only.ipynb`.


## 1. Setup and Configuration

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

# =============================================================================
# DIRECTORY PATHS
# =============================================================================
DATA_DIR       = Path(r"D:\StockTwits\Data\v1\data\csv")
MODEL_DATA_DIR = Path(r"D:\StockTwits\Data")
EMBED_FILE     = DATA_DIR / "text_embeddings_mlcrowd" / "text_embeddings_stock_day.pkl"
INPUT_DATA     = MODEL_DATA_DIR / "merged_master.pkl"
OUTPUT_FILE    = MODEL_DATA_DIR / "text_master.pkl"
META_FILE      = MODEL_DATA_DIR / "text_master.json"

print(f"Embeddings : {EMBED_FILE}")
print(f"Panel      : {INPUT_DATA}")
print(f"Output     : {OUTPUT_FILE}")


## 2. Load the Stock-Day Embeddings

In [ ]:
t0 = time.time()
emb = pd.read_pickle(EMBED_FILE)
EMBED_COLS = [c for c in emb.columns if c.startswith("embed_") and c != "embed_n"]
emb["date"] = pd.to_datetime(emb["date"])
emb["symbol"] = emb["symbol"].astype(str)
assert not emb.duplicated(["symbol", "date"]).any()
assert len(EMBED_COLS) == 384 and not emb[EMBED_COLS].isna().any().any()
print(f"Embeddings: {len(emb):,} stock-days x {len(EMBED_COLS)} dims, "
      f"{emb['date'].min().date()} to {emb['date'].max().date()}, {emb['symbol'].nunique():,} symbols "
      f"[{time.time() - t0:.0f}s]")


## 3. Load the Panel and Join

`merged_master` is keyed by (`ticker`, `date`) on the CRSP side, the embeddings by
(`symbol`, `date`); `perpare_training_data.ipynb` matches the two on `ticker == symbol` and so
does this cell. `mm_index` keeps the panel's row label so that predictions made from this table
can be placed back onto `merged_master` rows (the 04/05 notebooks index prediction files that way).


In [ ]:
t0 = time.time()
mm = pd.read_pickle(INPUT_DATA)
mm["date"] = pd.to_datetime(mm["date"])
mm.insert(0, "mm_index", mm.index.to_numpy())
print(f"Panel: {len(mm):,} rows x {mm.shape[1] - 1} columns, "
      f"{mm['date'].min().date()} to {mm['date'].max().date()} [{time.time() - t0:.0f}s]")

has_msgs = mm["log_volume"] > 0
text = mm.merge(emb.rename(columns={"symbol": "ticker"}), on=["ticker", "date"], how="inner")
del mm

n_emb_matched = text.drop_duplicates(["ticker", "date"]).shape[0]
print(f"Panel rows with >=1 message (log_volume > 0): {int(has_msgs.sum()):,} ({has_msgs.mean():.1%} of the panel)")
print(f"Panel rows with text (joined):               {len(text):,}")
print(f"Embedding stock-days matched to the panel:   {n_emb_matched:,} of {len(emb):,} "
      f"({len(emb) - n_emb_matched:,} outside the panel: 2024 and unmatched tickers)")
print(f"Duplicate (ticker, date) rows (two PERMNOs):  {int(text.duplicated(['ticker', 'date']).sum()):,}")
print(f"Rows with missing target f_cumret1:           {int(text['f_cumret1'].isna().sum()):,}")
del emb, has_msgs


## 4. Inspect

In [ ]:
text = text.sort_values(["date", "permno"]).reset_index(drop=True)
key_cols = ["mm_index", "permno", "ticker", "date", "f_cumret1"]
ar_cols = [c for c in text.columns if c.startswith("ar_")]
feat_cols = [c for c in text.columns if c not in key_cols + ar_cols + ["embed_n"] + EMBED_COLS]
text = text[key_cols + ar_cols + feat_cols + ["embed_n"] + EMBED_COLS]

print(f"Shape: {text.shape}  (~{text.memory_usage(deep=True).sum() / 1024**3:.2f} GB in memory)")
print(f"Columns: {len(key_cols)} keys/target, {len(ar_cols)} abnormal returns, {len(feat_cols)} features, "
      f"embed_n + {len(EMBED_COLS)} embedding dims")
print(f"Features: {feat_cols}")
print(f"Nulls in embedding columns: {int(text[EMBED_COLS].isna().sum().sum())}")
print(f"Messages per stock-day (embed_n): median {text['embed_n'].median():.0f}, mean {text['embed_n'].mean():.1f}, "
      f"max {text['embed_n'].max():,}")
print("\nRows per year:")
print(text.groupby(text["date"].dt.year).size().to_string())


## 5. Save

In [ ]:
print(f"Saving to {OUTPUT_FILE} ...")
text.to_pickle(OUTPUT_FILE)
meta = {"embeddings": str(EMBED_FILE), "panel": str(INPUT_DATA), "n_rows": int(len(text)),
        "n_columns": int(text.shape[1]), "embedding_columns": EMBED_COLS, "feature_columns": feat_cols,
        "abnormal_return_columns": ar_cols, "date_range": [str(text["date"].min().date()), str(text["date"].max().date())],
        "created": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}
META_FILE.write_text(json.dumps(meta, indent=2))
verify = pd.read_pickle(OUTPUT_FILE)
assert verify.shape == text.shape
print(f"Saved and verified: {OUTPUT_FILE.stat().st_size / 1024**3:.1f} GB; metadata in {META_FILE.name}")


## How the rest of the pipeline picks this up

- `03a - linear regression/prediction_linear_regression_text_only.ipynb` reads `text_master.pkl`
  and estimates the walk-forward regressions on the 384 embedding columns only. Its prediction
  file carries `index = mm_index`, so the 04/05 notebooks can place the predictions onto the
  panel exactly like the baseline files.
- The 53 baseline features are carried along so that "all features plus text" can be estimated
  on the same tweeted stock-days without materialising 384 columns on the full panel.
- Compressed representations (principal components, supervised scores) are deliberately not
  produced here; if needed they belong in a separate, explicitly walk-forward step.
